# 스마트 창고 출고 지연 예측 v8 (v6 기반)

### 핵심 분석: v6가 베스트인 이유
| 항목 | v6 | v7 |
|---|---|---|
| OOF MAE | **8.7129** | 8.7184 (더 나쁨) |
| Public Score | **베스트** | 하락 |
| Huber 모델 | ✅ OOF 8.7451 (최고!) | ❌ ExtraTrees로 교체 |
| Adversarial | ❌ 없음 | ✅ 추가했지만 악화 |

### v8 전략 (v6 베이스)
1. **v6 코어 유지** — Huber 모델 포함, Adversarial 미사용
2. **LGB Optuna 스킵** — v6 최적 파라미터 하드코딩
3. **CB Optuna 15 trials** — 빠른 튜닝
4. **XGB 수정** — MAE 목적함수 (RMSE→MAE), 조기종료
5. **Quantile LGB** (q=0.75, q=0.85) — 극단값 tail 직접 보정
6. **LGB Tweedie** — 왜도=5.68 비대칭 분포 특화
7. **CB Multi-seed** — CatBoost 다양성 증가
8. **피처 강화** — 복합 병목 지수, 다중 실패 카운트 등

## 0. 임포트 + 설정

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
POWER   = 0.4
np.random.seed(SEED)

# ──────────────────────────────────────────────────────────
# v6 Optuna 최적 파라미터 하드코딩 (27번 trial이 최적)
# LGB: n_estimators=2478, lr=0.0165, max_depth=7, num_leaves=61
# ──────────────────────────────────────────────────────────
BEST_LGB = dict(
    n_estimators      = 2478,
    learning_rate     = 0.016478723775160582,
    max_depth         = 7,
    num_leaves        = 61,
    subsample         = 0.7444928430592912,
    colsample_bytree  = 0.5056775143285802,
    reg_alpha         = 0.24040098526779663,
    reg_lambda        = 6.427489016616686,
    min_child_samples = 78,
)
print('환경 설정 완료 | v6 LGB 최적 파라미터 로드 완료')
print(f'LGB: n_est={BEST_LGB["n_estimators"]}, lr={BEST_LGB["learning_rate"]:.4f}, depth={BEST_LGB["max_depth"]}')

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


환경 설정 완료 | v6 LGB 최적 파라미터 로드 완료
LGB: n_est=2478, lr=0.0165, depth=7


## 1. 데이터 로드 + 레이아웃 처리

In [2]:
train  = pd.read_csv('./data/train.csv')
test   = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

layout_type_map = {v: i for i, v in enumerate(layout['layout_type'].unique())}
layout['layout_type_enc'] = layout['layout_type'].map(layout_type_map)

layout_feats = [c for c in layout.columns if c not in ['layout_id','layout_type','layout_type_enc']]
scaler = StandardScaler()
layout_scaled = scaler.fit_transform(layout[layout_feats].fillna(0))
km = KMeans(n_clusters=8, random_state=SEED, n_init=10)
layout['layout_cluster'] = km.fit_predict(layout_scaled)

train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout,  on='layout_id', how='left')

for mask, df in [(train['layout_cluster'].isna(), train),
                 (test['layout_cluster'].isna(),  test)]:
    if mask.any():
        for feat in layout_feats:
            if feat in df.columns:
                df.loc[mask, feat] = df.loc[mask, feat].fillna(df[feat].median())
        df.loc[mask, 'layout_cluster']  = 8
        df.loc[mask, 'layout_type_enc'] = -1

print(f'train: {train.shape}, test: {test.shape}')

train: (250000, 110), test: (50000, 109)


## 2. 피처 엔지니어링 (v6 + v8 신규)

**v8 신규 피처**:
- `multi_bottleneck_score`: 팩포화+충전압박+사고+독포화 동시 발생 카운트 (0~4)
- `extreme_congestion_flag`: 혼잡>0.8 & 팩>0.9 동시 포화 (극단 지연 핵심 조건)
- `compounding_bottleneck`: 팩오버플로우 × 혼잡 × 로봇당 주문 (복합 압박 지수)
- `robot_health_index`: 배터리 × 저배터리율 × 활성 비율 (로봇 종합 건강)
- `pack_dock_bottleneck`: 팩 × 독 이중 포화
- `global_load_stress`: 전체 부하 ÷ 로봇 건강 (시스템 전체 스트레스)

In [3]:
def engineer_features(df):
    d = df.copy()
    # ── 로봇 압박 (v6) ──
    d['charge_pressure']       = d['charge_queue_length'] / (d['charger_count'] + 1)
    d['active_robot_ratio']    = d['robot_active'] / (d['robot_total'] + 1)
    d['idle_robot_ratio']      = d['robot_idle'] / (d['robot_total'] + 1)
    d['low_batt_robot_count']  = d['low_battery_ratio'] * d['robot_active']
    d['battery_stress']        = (1 - d['battery_mean'] / 100) * d['battery_std']
    d['effective_robot_avail'] = d['robot_total'] - d['low_batt_robot_count'] - d['charge_queue_length']
    d['robot_fault_ratio']     = d['fault_count_15m'] / (d['robot_active'] + 1)
    # ── 병목 복합 (v6) ──
    d['incident_total_15m']    = d['blocked_path_15m'] + d['near_collision_15m'] + d['fault_count_15m']
    d['congestion_load']       = d['congestion_score'] * d['avg_trip_distance']
    d['wait_per_intersection'] = d['intersection_wait_time_avg'] / (d['intersection_count'] + 1)
    d['path_congestion_gap']   = d['path_optimization_score'] - d['congestion_score']
    d['aisle_density']         = d['aisle_traffic_score'] / (d['aisle_width_avg'] + 0.1)
    # ── 주문 부하 (v6) ──
    d['order_complexity']      = d['unique_sku_15m'] * d['avg_items_per_order']
    d['urgent_heavy_ratio']    = d['urgent_order_ratio'] * d['heavy_item_ratio']
    d['effective_order_load']  = d['order_inflow_15m'] * (1 + d['urgent_order_ratio'])
    d['rework_pressure']       = d['return_order_ratio'] + d['replenishment_overlap']
    d['pick_complexity']       = d['pick_list_length_avg'] * (1 - d['sku_concentration'])
    # ── 설비 압박 (v6) ──
    d['dock_pack_util_avg']      = (d['pack_utilization'] + d['loading_dock_util'] + d['staging_area_util']) / 3
    d['orders_per_pack_station'] = d['order_inflow_15m'] / (d['pack_station_count'] + 1)
    d['orders_per_robot']        = d['order_inflow_15m'] / (d['robot_active'] + 1)
    d['robot_density']           = d['robot_total'] / (d['floor_area_sqm'] + 1)
    d['pack_area_pressure']      = d['pack_utilization'] * d['layout_compactness']
    d['dock_truck_bottleneck']   = d['loading_dock_util'] * d['outbound_truck_wait_min']
    # ── IT/환경 (v6) ──
    d['it_bottleneck']         = d['wms_response_time_ms'] * (1 + d['scanner_error_rate'])
    d['barcode_fail_rate']     = 1 - d['barcode_read_success_rate']
    d['label_scan_bottleneck'] = d['label_print_queue'] * (1 + d['scanner_error_rate'])
    d['conveyor_load']         = d['avg_package_weight_kg'] / (d['conveyor_speed_mps'] + 0.01)
    # ── 인력 (v6) ──
    d['orders_per_staff']      = d['order_inflow_15m'] / (d['staff_on_floor'] + 1)
    d['shift_load_ratio']      = d['order_inflow_15m'] / (d['prev_shift_volume'] + 1)
    d['handover_pressure']     = d['shift_handover_delay_min'] * d['prev_shift_volume'] / 1000
    # ── 시간 (v6) ──
    d['shift_hour_sin'] = np.sin(2 * np.pi * d['shift_hour'] / 24)
    d['shift_hour_cos'] = np.cos(2 * np.pi * d['shift_hour'] / 24)
    d['dow_sin']        = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']        = np.cos(2 * np.pi * d['day_of_week'] / 7)
    d['is_peak_hour']   = d['shift_hour'].isin([9,10,11,14,15,16,17]).astype(int)
    # ── 레이아웃 교호 (v6) ──
    d['area_per_robot']          = d['floor_area_sqm'] / (d['robot_total'] + 1)
    d['crossdock_dock_pressure'] = d['cross_dock_ratio'] * d['loading_dock_util']
    d['robot_per_pack_station']  = d['robot_total'] / (d['pack_station_count'] + 1)
    # ── 핵심 교호작용 (v6) ──
    d['robot_order_congestion']  = d['orders_per_robot'] * d['congestion_score']
    d['battery_order_pressure']  = d['battery_stress'] * d['effective_order_load']
    d['low_batt_x_congestion']   = d['low_battery_ratio'] * d['congestion_score']
    d['charge_q_x_orders']       = d['charge_queue_length'] * d['order_inflow_15m']
    # ── pack 비선형 (v6) ──
    d['pack_saturated']         = (d['pack_utilization'] > 0.95).astype(int)
    d['pack_overflow_pressure'] = np.maximum(d['pack_utilization'] - 0.95, 0) * d['order_inflow_15m']
    d['pack_util_sq']           = d['pack_utilization'] ** 2
    # ── 비율 피처 (v6) ──
    d['order_robot_balance']   = d['order_inflow_15m'] / (d['robot_active'] * d['robot_utilization'] + 1)
    d['charge_demand_ratio']   = d['charge_queue_length'] / (d['robot_charging'] + 1)
    d['pack_order_ratio']      = d['pack_utilization'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['congestion_per_robot']  = d['congestion_score'] / (d['robot_active'] + 1)
    d['fault_per_100_orders']  = d['fault_count_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['block_per_100_orders']  = d['blocked_path_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['dock_order_match']      = d['loading_dock_util'] / (d['order_inflow_15m'] / 200 + 0.01)
    d['staff_order_match']     = d['staff_on_floor'] / (d['order_inflow_15m'] / 50 + 0.01)
    d['battery_adequacy']      = d['battery_mean'] / (100 - d['low_battery_ratio'] * 100 + 1)
    d['trip_efficiency']       = d['avg_trip_distance'] / (d['robot_utilization'] + 0.01)

    # ════════════════════════════════════════
    # [v8] 신규 피처 — 극단값 예측 강화
    # ════════════════════════════════════════
    # 다중 병목 카운트 (극단 지연은 여러 병목이 동시 발생)
    d['multi_bottleneck_score'] = (
        d['pack_saturated'] +
        (d['charge_pressure'] > 3).astype(int) +
        (d['incident_total_15m'] > 3).astype(int) +
        (d['loading_dock_util'] > 0.9).astype(int)
    )
    # 극단 혼잡 플래그 (혼잡 + 팩 동시 포화)
    d['extreme_congestion_flag'] = (
        (d['congestion_score'] > 0.8) & (d['pack_utilization'] > 0.9)
    ).astype(int)
    # 복합 압박 지수 (팩오버플로우 × 혼잡 × 로봇당 주문)
    d['compounding_bottleneck'] = (
        d['pack_overflow_pressure'] * d['congestion_score'] * d['orders_per_robot']
    )
    # 로봇 건강 종합 지수
    d['robot_health_index'] = (
        (d['battery_mean'] / 100) * (1 - d['low_battery_ratio']) * d['active_robot_ratio']
    )
    # 팩 + 독 이중 병목
    d['pack_dock_bottleneck'] = d['pack_utilization'] * d['loading_dock_util']
    # 전체 부하 스트레스 (로봇 건강 반영)
    d['global_load_stress'] = (
        d['effective_order_load'] * d['congestion_load'] / (d['robot_health_index'] + 0.01)
    )
    # 독 복합 압박
    d['dock_pressure_composite'] = (
        d['dock_truck_bottleneck'] * d['effective_order_load'] / (d['pack_station_count'] + 1)
    )
    return d

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f'피처 엔지니어링 완료: {train_fe.shape}')

피처 엔지니어링 완료: (250000, 172)


## 3. 래그 + 롤링 피처

In [4]:
LAG_COLS = [
    'order_inflow_15m','congestion_score','robot_utilization','battery_mean',
    'charge_queue_length','blocked_path_15m','fault_count_15m',
    'pack_utilization','loading_dock_util','effective_order_load','orders_per_robot',
    'multi_bottleneck_score','robot_health_index',  # [v8]
]
CUM_COLS  = ['order_inflow_15m','congestion_score','blocked_path_15m','fault_count_15m','incident_total_15m']
ROLL_COLS = ['order_inflow_15m','congestion_score','battery_mean','pack_utilization',
             'robot_utilization','loading_dock_util','multi_bottleneck_score']  # [v8]

def add_temporal(df, lag_cols, cum_cols, roll_cols):
    d = df.copy().sort_values(['scenario_id','shift_hour']).reset_index(drop=True)
    d['slot_idx']      = d.groupby('scenario_id').cumcount()
    d['slot_progress'] = d['slot_idx'] / 24
    for col in lag_cols:
        if col not in d.columns: continue
        l1 = d.groupby('scenario_id')[col].shift(1)
        l2 = d.groupby('scenario_id')[col].shift(2)
        d[f'{col}_lag1']       = l1
        d[f'{col}_lag2']       = l2
        d[f'{col}_diff1']      = d[col] - l1
        d[f'{col}_pct_change'] = (d[col] - l1) / (l1.abs() + 1)
    for col in cum_cols:
        if col not in d.columns: continue
        d[f'{col}_cumsum'] = d.groupby('scenario_id')[col].cumsum()
    for col in roll_cols:
        if col not in d.columns: continue
        d[f'{col}_roll3_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).mean())
        d[f'{col}_roll3_std']  = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).std())
    for col in ['order_inflow_15m','congestion_score','pack_utilization','battery_mean']:
        if col not in d.columns: continue
        d[f'{col}_exp_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.expanding().mean())
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        if col not in d.columns: continue
        def rolling_slope(x):
            out = np.full(len(x), np.nan); vals = x.values
            for i in range(2, len(vals)):
                yv = vals[i-2:i+1]
                if np.all(np.isfinite(yv)): out[i] = (yv[2]-yv[0])/2
            return pd.Series(out, index=x.index)
        d[f'{col}_trend3'] = d.groupby('scenario_id')[col].transform(rolling_slope)
    if 'pack_utilization_lag1' in d.columns:
        d['pack_sat_lag1'] = (d['pack_utilization_lag1'] > 0.95).astype(int)
    if 'order_inflow_15m_cumsum' in d.columns:
        d['cum_order_per_robot'] = d['order_inflow_15m_cumsum'] / (d['robot_total'] + 1)
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        rm = f'{col}_roll3_mean'
        if rm in d.columns:
            d[f'{col}_deviation']       = d[col] - d[rm]
            d[f'{col}_deviation_ratio'] = d[f'{col}_deviation'] / (d[rm].abs() + 1)
    # [v8] slot_progress 교호작용
    d['slot_progress_x_order_load']  = d['slot_progress'] * d['effective_order_load']
    d['slot_progress_x_congestion']  = d['slot_progress'] * d['congestion_score']
    d['slot_progress_x_pack_util']   = d['slot_progress'] * d['pack_utilization']

    fill_cols = [c for c in d.columns if any(t in c for t in
                 ['_lag','_diff','_cumsum','_roll3','_exp_','_trend','_sat_lag',
                  '_pct_change','_deviation','slot_progress_x'])]
    for col in fill_cols:
        d[col] = d.groupby('scenario_id')[col].transform(lambda x: x.fillna(x.median()))
    d[fill_cols] = d[fill_cols].fillna(0)
    return d

train_fe = add_temporal(train_fe, LAG_COLS, CUM_COLS, ROLL_COLS)
test_fe  = add_temporal(test_fe,  LAG_COLS, CUM_COLS, ROLL_COLS)
print(f'전체 피처: {len([c for c in train_fe.columns if c not in ID_COLS+[TARGET]])}')

전체 피처: 259


## 4. 시나리오 통계 (최소) + 타겟 인코딩

In [5]:
SC_COLS = [
    'order_inflow_15m','congestion_score','robot_utilization',
    'battery_mean','pack_utilization','loading_dock_util',
]
for col in SC_COLS:
    if col not in train_fe.columns: continue
    for df in [train_fe, test_fe]:
        df[f'sc_{col}_mean'] = df.groupby('scenario_id')[col].transform('mean')
        df[f'sc_{col}_dev']  = df[col] - df[f'sc_{col}_mean']

# [v8] 시나리오 수준 교호작용
for df in [train_fe, test_fe]:
    df['sc_double_bottleneck'] = df['sc_pack_utilization_mean'] * df['sc_congestion_score_mean']
    df['sc_pack_x_progress']   = df['sc_pack_utilization_mean'] * df['slot_progress']
    df['sc_congestion_x_prog'] = df['sc_congestion_score_mean'] * df['slot_progress']

def target_encode_cv(tr, te, col, target, n_splits=5, alpha=50):
    gm = tr[target].mean(); enc = np.zeros(len(tr))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for ti, vi in kf.split(tr):
        s = tr.iloc[ti].groupby(col)[target].agg(['mean','count'])
        s['sm'] = (s['count']*s['mean']+alpha*gm)/(s['count']+alpha)
        enc[vi] = tr.iloc[vi][col].map(s['sm']).fillna(gm)
    fs = tr.groupby(col)[target].agg(['mean','count'])
    fs['sm'] = (fs['count']*fs['mean']+alpha*gm)/(fs['count']+alpha)
    return enc, te[col].map(fs['sm']).fillna(gm).values

for col_name, feat_name in [('layout_cluster','te_layout_cluster'),('layout_type_enc','te_layout_type')]:
    tr_e, te_e = target_encode_cv(train_fe, test_fe, col_name, TARGET)
    train_fe[feat_name] = tr_e; test_fe[feat_name] = te_e
print('시나리오 통계 + 타겟 인코딩 완료')

시나리오 통계 + 타겟 인코딩 완료


## 5. 피처 준비

In [6]:
feature_cols = [
    c for c in train_fe.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(train_fe[c])
]
for col in feature_cols:
    med = train_fe[col].median()
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col]  = test_fe[col].fillna(med)
drop_cols = [c for c in feature_cols
             if train_fe[c].nunique() <= 1 or train_fe[c].std() < 1e-10]
feature_cols = [c for c in feature_cols if c not in drop_cols]
print(f'최종 피처: {len(feature_cols)}개 (제거: {len(drop_cols)}개)')

X      = train_fe[feature_cols].copy()
y      = train_fe[TARGET].copy()
X_test = test_fe[feature_cols].copy()

y_tr_sqrt  = np.sqrt(y.clip(lower=0))
y_tr_log   = np.log1p(y.clip(lower=0))
y_tr_power = np.power(y.clip(lower=0) + 1e-8, POWER)

def decode_sqrt(p):  return (p.clip(0))**2
def decode_log(p):   return np.expm1(p).clip(0)
def decode_power(p): return np.power(p.clip(0), 1/POWER)

gkf    = GroupKFold(n_splits=N_FOLDS)
groups = train_fe['scenario_id']
print(f'왜도: 원본={stats.skew(y):.2f}, sqrt={stats.skew(y_tr_sqrt):.2f}, '
      f'log={stats.skew(y_tr_log):.2f}, power={stats.skew(y_tr_power):.2f}')

최종 피처: 275개 (제거: 0개)
왜도: 원본=5.68, sqrt=1.47, log=0.08, power=0.96


## 6. CB Optuna 튜닝 (15 trials)

In [7]:
tidx = np.random.choice(len(X), 50000, replace=False)
Xt   = X.iloc[tidx].reset_index(drop=True)
yt_s = y_tr_sqrt.iloc[tidx].reset_index(drop=True)
yt_r = y.iloc[tidx].reset_index(drop=True)
gt   = groups.iloc[tidx].reset_index(drop=True)

def cb_obj(trial):
    p = dict(
        iterations       = trial.suggest_int('iterations', 500, 2500),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        depth            = trial.suggest_int('depth', 4, 8),
        l2_leaf_reg      = trial.suggest_float('l2_leaf_reg', 1.0, 50, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 0.9),
        min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 50, 500),
    )
    F = dict(loss_function='MAE', eval_metric='MAE', bootstrap_type='Bernoulli',
             random_seed=SEED, early_stopping_rounds=30, verbose=0,
             task_type='GPU', devices='0')
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = CatBoostRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti], eval_set=(Xt.iloc[vi], yt_s.iloc[vi]))
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('CB Optuna (15 trials)...')
study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(cb_obj, n_trials=15, show_progress_bar=True)
BEST_CB = study_cb.best_params
print(f'CB 최적 MAE: {study_cb.best_value:.4f}')
print(f'파라미터: {BEST_CB}')

CB Optuna (15 trials)...


  0%|          | 0/15 [00:00<?, ?it/s]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91618:   7%|▋         | 1/15 [01:05<15:21, 65.83s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91618:  13%|█▎        | 2/15 [02:16<14:56, 68.95s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Best trial: 0. Best value: 8.91618:  20%|██        | 3/15 [03:27<13:56, 69.69s/it]Default metric period is 5 because MAE is/are not implemented for GPU
Default metric p

CB 최적 MAE: 8.9143
파라미터: {'iterations': 2145, 'learning_rate': 0.028396055665149623, 'depth': 6, 'l2_leaf_reg': 3.8665470910741826, 'subsample': 0.8086656573027753, 'min_data_in_leaf': 145}


## 7. M1: LGB (sqrt)

In [8]:
oof_lgb_sqrt, test_lgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
lgb_imp = np.zeros(len(feature_cols))
print('M1: LGB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)])
    oof_lgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    lgb_imp         += m.feature_importances_ / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_sqrt[vi]):.4f}')
print(f'M1 OOF MAE: {mean_absolute_error(y, oof_lgb_sqrt):.4f}')

M1: LGB(sqrt) 5-Fold
-- Fold 1 --
[200]	valid_0's l1: 0.928666
[400]	valid_0's l1: 0.921728
[600]	valid_0's l1: 0.92014
   MAE: 8.8296
-- Fold 2 --
[200]	valid_0's l1: 0.926729
[400]	valid_0's l1: 0.919709
   MAE: 8.7586
-- Fold 3 --
[200]	valid_0's l1: 0.978769
[400]	valid_0's l1: 0.972021
   MAE: 9.3252
-- Fold 4 --
[200]	valid_0's l1: 0.90226
[400]	valid_0's l1: 0.892704
[600]	valid_0's l1: 0.891107
   MAE: 8.3120
-- Fold 5 --
[200]	valid_0's l1: 0.915308
[400]	valid_0's l1: 0.907859
   MAE: 8.6789
M1 OOF MAE: 8.7809


## 8. M2: CB (sqrt)

In [9]:
oof_cb_sqrt, test_cb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M2: CB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=200,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=(X.iloc[vi], y_tr_sqrt.iloc[vi]))
    oof_cb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_cb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_cb_sqrt[vi]):.4f}')
print(f'M2 OOF MAE: {mean_absolute_error(y, oof_cb_sqrt):.4f}')

M2: CB(sqrt) 5-Fold
-- Fold 1 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7174016	test: 1.7208895	best: 1.7208895 (0)	total: 8.66ms	remaining: 18.6s
200:	learn: 1.0105501	test: 1.0160447	best: 1.0160447 (200)	total: 1.09s	remaining: 10.6s
400:	learn: 0.9410376	test: 0.9521286	best: 0.9521286 (400)	total: 2.17s	remaining: 9.42s
600:	learn: 0.9207983	test: 0.9382627	best: 0.9382627 (600)	total: 3.2s	remaining: 8.22s
800:	learn: 0.9054627	test: 0.9337501	best: 0.9337501 (800)	total: 4.25s	remaining: 7.14s
1000:	learn: 0.8915520	test: 0.9318412	best: 0.9318412 (1000)	total: 5.33s	remaining: 6.09s
1200:	learn: 0.8792926	test: 0.9306245	best: 0.9306245 (1200)	total: 6.41s	remaining: 5.04s
1400:	learn: 0.8684702	test: 0.9290009	best: 0.9289671 (1385)	total: 7.48s	remaining: 3.97s
1600:	learn: 0.8581930	test: 0.9279781	best: 0.9279662 (1599)	total: 8.58s	remaining: 2.91s
1800:	learn: 0.8482881	test: 0.9268698	best: 0.9268535 (1796)	total: 9.66s	remaining: 1.84s
2000:	learn: 0.8381530	test: 0.9259455	best: 0.9259356 (1997)	total: 10.7s	remaining: 773ms
be

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7194686	test: 1.7124573	best: 1.7124573 (0)	total: 5.97ms	remaining: 12.8s
200:	learn: 1.0126592	test: 1.0105390	best: 1.0105390 (200)	total: 1.07s	remaining: 10.4s
400:	learn: 0.9415200	test: 0.9484405	best: 0.9484405 (400)	total: 2.16s	remaining: 9.39s
600:	learn: 0.9212945	test: 0.9358457	best: 0.9358457 (600)	total: 3.23s	remaining: 8.3s
800:	learn: 0.9058659	test: 0.9295088	best: 0.9295088 (800)	total: 4.37s	remaining: 7.34s
1000:	learn: 0.8922415	test: 0.9275265	best: 0.9274752 (995)	total: 5.46s	remaining: 6.24s
1200:	learn: 0.8804209	test: 0.9265075	best: 0.9264075 (1192)	total: 6.55s	remaining: 5.15s
1400:	learn: 0.8694138	test: 0.9253177	best: 0.9252563 (1394)	total: 7.62s	remaining: 4.05s
1600:	learn: 0.8594137	test: 0.9242364	best: 0.9242290 (1572)	total: 8.69s	remaining: 2.95s
1800:	learn: 0.8492127	test: 0.9234970	best: 0.9234970 (1800)	total: 9.78s	remaining: 1.87s
bestTest = 0.9231734375
bestIteration = 1908
Shrink model to first 1909 iterations.
   MAE: 8.7

...

0:	learn: 1.7147828	test: 1.7315706	best: 1.7315706 (0)	total: 6.82ms	remaining: 14.6s
200:	learn: 0.9982665	test: 1.0662264	best: 1.0662264 (200)	total: 1.09s	remaining: 10.5s
400:	learn: 0.9271212	test: 1.0094644	best: 1.0094644 (400)	total: 2.13s	remaining: 9.28s
600:	learn: 0.9077138	test: 0.9949196	best: 0.9949196 (600)	total: 3.18s	remaining: 8.18s
800:	learn: 0.8930372	test: 0.9890929	best: 0.9890929 (800)	total: 4.26s	remaining: 7.16s
1000:	learn: 0.8802473	test: 0.9860490	best: 0.9860204 (989)	total: 5.35s	remaining: 6.11s
1200:	learn: 0.8680893	test: 0.9841828	best: 0.9841828 (1200)	total: 6.44s	remaining: 5.06s
1400:	learn: 0.8574855	test: 0.9818670	best: 0.9818670 (1400)	total: 7.51s	remaining: 3.99s
1600:	learn: 0.8471150	test: 0.9803778	best: 0.9803778 (1600)	total: 8.59s	remaining: 2.92s
1800:	learn: 0.8373028	test: 0.9790807	best: 0.9790807 (1800)	total: 9.67s	remaining: 1.85s
2000:	learn: 0.8285945	test: 0.9769752	best: 0.9769752 (2000)	total: 10.8s	remaining: 776ms
21

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7196591	test: 1.7117892	best: 1.7117892 (0)	total: 6.01ms	remaining: 12.9s
200:	learn: 1.0176165	test: 0.9959466	best: 0.9959466 (200)	total: 1.1s	remaining: 10.7s
400:	learn: 0.9490227	test: 0.9217796	best: 0.9217796 (400)	total: 2.15s	remaining: 9.34s
600:	learn: 0.9286379	test: 0.9065098	best: 0.9065098 (600)	total: 3.19s	remaining: 8.19s
800:	learn: 0.9133716	test: 0.9009227	best: 0.9009227 (800)	total: 4.25s	remaining: 7.13s
1000:	learn: 0.8991563	test: 0.8978105	best: 0.8978105 (1000)	total: 5.33s	remaining: 6.09s
1200:	learn: 0.8872887	test: 0.8953126	best: 0.8953126 (1200)	total: 6.39s	remaining: 5.02s
bestTest = 0.8944313281
bestIteration = 1282
Shrink model to first 1283 iterations.
   MAE: 8.3322
-- Fold 5 --


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.7189395	test: 1.7148664	best: 1.7148664 (0)	total: 6.27ms	remaining: 13.4s
200:	learn: 1.0157581	test: 0.9962284	best: 0.9962284 (200)	total: 1.08s	remaining: 10.4s
400:	learn: 0.9458859	test: 0.9297835	best: 0.9297835 (400)	total: 2.13s	remaining: 9.26s
600:	learn: 0.9252009	test: 0.9178631	best: 0.9178631 (600)	total: 3.2s	remaining: 8.22s
800:	learn: 0.9099238	test: 0.9132787	best: 0.9132745 (799)	total: 4.26s	remaining: 7.15s
1000:	learn: 0.8969039	test: 0.9110078	best: 0.9109387 (998)	total: 5.33s	remaining: 6.09s
1200:	learn: 0.8856782	test: 0.9090913	best: 0.9090913 (1200)	total: 6.4s	remaining: 5.03s
bestTest = 0.9086227344
bestIteration = 1269
Shrink model to first 1270 iterations.
   MAE: 8.6770
M2 OOF MAE: 8.8034


## 9. M3: XGB (MAE 수정) ← v8 핵심 수정
v6: RMSE 최적화 → v8: `objective='reg:absoluteerror'` + `eval_metric='mae'` + 조기종료

In [10]:
oof_xgb_mae, test_xgb_mae = np.zeros(len(X)), np.zeros(len(X_test))
print('M3: XGB(MAE) 5-Fold  ← v8 수정')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = XGBRegressor(
        n_estimators=3000, learning_rate=0.02, max_depth=6,
        subsample=0.7, colsample_bytree=0.5,
        reg_alpha=5, reg_lambda=5, min_child_weight=100,
        objective='reg:absoluteerror',   # MAE 최적화
        eval_metric='mae',
        early_stopping_rounds=50,
        device='cuda', tree_method='hist',
        random_state=SEED, verbosity=0
    )
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          verbose=300)
    oof_xgb_mae[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_xgb_mae   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_xgb_mae[vi]):.4f}')
print(f'M3 OOF MAE: {mean_absolute_error(y, oof_xgb_mae):.4f}')

M3: XGB(MAE) 5-Fold  ← v8 수정
-- Fold 1 --
[0]	validation_0-mae:1.70555
[300]	validation_0-mae:0.92028
[600]	validation_0-mae:0.91869
[900]	validation_0-mae:0.91810
[936]	validation_0-mae:0.91817
   MAE: 8.8094
-- Fold 2 --
[0]	validation_0-mae:1.69697
[300]	validation_0-mae:0.92065
[600]	validation_0-mae:0.91922
[650]	validation_0-mae:0.91944
   MAE: 8.7560
-- Fold 3 --
[0]	validation_0-mae:1.71630
[300]	validation_0-mae:0.97212
[438]	validation_0-mae:0.97105
   MAE: 9.3175
-- Fold 4 --
[0]	validation_0-mae:1.69679
[300]	validation_0-mae:0.89348
[524]	validation_0-mae:0.89173
   MAE: 8.3196
-- Fold 5 --
[0]	validation_0-mae:1.69953
[300]	validation_0-mae:0.90824
[477]	validation_0-mae:0.90744
   MAE: 8.6755
M3 OOF MAE: 8.7756


## 10. M4: LGB (log1p)

In [11]:
oof_lgb_log, test_lgb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M4: LGB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_log.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_log.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_lgb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_log[vi]):.4f}')
print(f'M4 OOF MAE: {mean_absolute_error(y, oof_lgb_log):.4f}')

M4: LGB(log) 5-Fold
  Fold 1 MAE: 8.8230
  Fold 2 MAE: 8.7352
  Fold 3 MAE: 9.2784
  Fold 4 MAE: 8.3347
  Fold 5 MAE: 8.6721
M4 OOF MAE: 8.7687


## 11. M5: CB (log1p)

In [12]:
oof_cb_log, test_cb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M5: CB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bernoulli', random_seed=SEED,
                          early_stopping_rounds=50, verbose=0,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_log.iloc[ti], eval_set=(X.iloc[vi], y_tr_log.iloc[vi]))
    oof_cb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_cb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_cb_log[vi]):.4f}')
print(f'M5 OOF MAE: {mean_absolute_error(y, oof_cb_log):.4f}')

M5: CB(log) 5-Fold


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 1 MAE: 8.8400


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 2 MAE: 8.7377


...

  Fold 3 MAE: 9.2733


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 4 MAE: 8.2953


Default metric period is 5 because MAE is/are not implemented for GPU


  Fold 5 MAE: 8.6655
M5 OOF MAE: 8.7624


## 12. M6: LGB (power=0.4)

In [13]:
oof_lgb_pow, test_lgb_pow = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M6: LGB(power={POWER}) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_power.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_power.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_pow[vi] = decode_power(m.predict(X.iloc[vi]))
    test_lgb_pow   += decode_power(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_pow[vi]):.4f}')
print(f'M6 OOF MAE: {mean_absolute_error(y, oof_lgb_pow):.4f}')

M6: LGB(power=0.4) 5-Fold
  Fold 1 MAE: 8.8233
  Fold 2 MAE: 8.7358
  Fold 3 MAE: 9.2992
  Fold 4 MAE: 8.3067
  Fold 5 MAE: 8.6742
M6 OOF MAE: 8.7678


## 13. M7: LGB (Huber, delta=1.5) ← v6 베스트 단일 모델!
**v6에서 OOF MAE 8.7451로 개별 모델 중 최고 성능 — v7에서 잘못 제거됨**

In [14]:
oof_lgb_huber, test_lgb_huber = np.zeros(len(X)), np.zeros(len(X_test))
print('M7: LGB(Huber, delta=1.5) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='huber', huber_delta=1.5,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_huber[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_huber   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_huber[vi]):.4f}')
print(f'M7 OOF MAE: {mean_absolute_error(y, oof_lgb_huber):.4f}')

M7: LGB(Huber, delta=1.5) 5-Fold
  Fold 1 MAE: 8.8008
  Fold 2 MAE: 8.7302
  Fold 3 MAE: 9.2639
  Fold 4 MAE: 8.2600
  Fold 5 MAE: 8.6493
M7 OOF MAE: 8.7408


## 14. M8: LGB Multi-seed (3 seeds)

In [15]:
SEEDS = [42, 123, 456]
oof_lgb_ms, test_lgb_ms = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M8: LGB Multi-seed ({len(SEEDS)} seeds)')
for seed in SEEDS:
    oof_tmp, test_tmp = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu',
                          random_state=seed, verbose=-1,
                          subsample_seed=seed, feature_fraction_seed=seed)
        m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
              eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_tmp[vi] = decode_sqrt(m.predict(X.iloc[vi]))
        test_tmp   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    oof_lgb_ms  += oof_tmp  / len(SEEDS)
    test_lgb_ms += test_tmp / len(SEEDS)
    print(f'  Seed {seed}: OOF MAE = {mean_absolute_error(y, oof_tmp):.4f}')
print(f'M8 OOF MAE: {mean_absolute_error(y, oof_lgb_ms):.4f}')

M8: LGB Multi-seed (3 seeds)
  Seed 42: OOF MAE = 8.7715
  Seed 123: OOF MAE = 8.7665
  Seed 456: OOF MAE = 8.7672
M8 OOF MAE: 8.7627


## 15. M9: LGB Quantile q=0.75 ← v8 신규
**극단값 tail 과소예측 직접 보정** — 75th percentile 조건부 예측으로 고지연 캡처

In [16]:
oof_lgb_q75, test_lgb_q75 = np.zeros(len(X)), np.zeros(len(X_test))
print('M9: LGB Quantile q=0.75  [v8 신규]')
Q75_PARAMS = dict(n_estimators=2000, learning_rate=0.015, max_depth=7, num_leaves=70,
                  subsample=0.74, colsample_bytree=0.5, reg_alpha=0.3, reg_lambda=7.0,
                  min_child_samples=80)
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**Q75_PARAMS, objective='quantile', alpha=0.75,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y.iloc[ti],
          eval_set=[(X.iloc[vi], y.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_q75[vi] = m.predict(X.iloc[vi]).clip(0)
    test_lgb_q75   += m.predict(X_test).clip(0) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_q75[vi]):.4f}  '
          f'95th={np.percentile(oof_lgb_q75[vi],95):.1f}')
print(f'M9 OOF MAE: {mean_absolute_error(y, oof_lgb_q75):.4f}')
print(f'  q75 예측 95th: {np.percentile(oof_lgb_q75,95):.1f}, 99th: {np.percentile(oof_lgb_q75,99):.1f}')

M9: LGB Quantile q=0.75  [v8 신규]
  Fold 1 MAE: 10.4888  95th=51.8
  Fold 2 MAE: 10.6594  95th=53.2
  Fold 3 MAE: 11.0595  95th=53.1
  Fold 4 MAE: 9.8955  95th=51.2
  Fold 5 MAE: 10.7084  95th=52.5
M9 OOF MAE: 10.5623
  q75 예측 95th: 52.4, 99th: 96.8


## 16. M10: LGB Quantile q=0.85 ← v8 신규
**극단값 tail 포착 강화** — 상위 15% 고지연 예측력 극대화

In [17]:
oof_lgb_q85, test_lgb_q85 = np.zeros(len(X)), np.zeros(len(X_test))
print('M10: LGB Quantile q=0.85  [v8 신규]')
Q85_PARAMS = dict(n_estimators=2000, learning_rate=0.015, max_depth=7, num_leaves=70,
                  subsample=0.74, colsample_bytree=0.5, reg_alpha=0.3, reg_lambda=7.0,
                  min_child_samples=80)
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**Q85_PARAMS, objective='quantile', alpha=0.85,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y.iloc[ti],
          eval_set=[(X.iloc[vi], y.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_q85[vi] = m.predict(X.iloc[vi]).clip(0)
    test_lgb_q85   += m.predict(X_test).clip(0) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_q85[vi]):.4f}  '
          f'95th={np.percentile(oof_lgb_q85[vi],95):.1f}')
print(f'M10 OOF MAE: {mean_absolute_error(y, oof_lgb_q85):.4f}')

M10: LGB Quantile q=0.85  [v8 신규]
  Fold 1 MAE: 13.2633  95th=64.2
  Fold 2 MAE: 13.2903  95th=66.0
  Fold 3 MAE: 13.9522  95th=66.1
  Fold 4 MAE: 12.4889  95th=62.9
  Fold 5 MAE: 13.6385  95th=65.7
M10 OOF MAE: 13.3266


## 17. M11: LGB Tweedie (power=1.5) ← v8 신규
**왜도=5.68 비대칭 분포 특화** — 물류 지연/보험 분야에서 검증된 목적함수

In [18]:
oof_lgb_tw, test_lgb_tw = np.zeros(len(X)), np.zeros(len(X_test))
print('M11: LGB Tweedie(power=1.5)  [v8 신규]')
TW_PARAMS = dict(n_estimators=2500, learning_rate=0.015, max_depth=7, num_leaves=70,
                 subsample=0.74, colsample_bytree=0.5, reg_alpha=0.25, reg_lambda=6.5,
                 min_child_samples=80)
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**TW_PARAMS, objective='tweedie', tweedie_variance_power=1.5,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y.iloc[ti].clip(lower=0.01),
          eval_set=[(X.iloc[vi], y.iloc[vi].clip(lower=0.01))],
          callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_tw[vi] = m.predict(X.iloc[vi]).clip(0)
    test_lgb_tw   += m.predict(X_test).clip(0) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_tw[vi]):.4f}')
print(f'M11 OOF MAE: {mean_absolute_error(y, oof_lgb_tw):.4f}')
print(f'  Tweedie 99th: {np.percentile(oof_lgb_tw,99):.1f}')

M11: LGB Tweedie(power=1.5)  [v8 신규]
  Fold 1 MAE: 9.0148
  Fold 2 MAE: 8.9292
  Fold 3 MAE: 9.5599
  Fold 4 MAE: 8.4169
  Fold 5 MAE: 8.9704
M11 OOF MAE: 8.9782
  Tweedie 99th: 70.0


## 18. M12: CB Multi-seed (3 seeds) ← v8 신규
**CatBoost 다양성 증가** — 단일 seed보다 분산 감소

In [19]:
CB_SEEDS = [42, 777, 2025]
oof_cb_ms, test_cb_ms = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M12: CB Multi-seed ({len(CB_SEEDS)} seeds)  [v8 신규]')
for seed in CB_SEEDS:
    oof_tmp, test_tmp = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                              bootstrap_type='Bernoulli', random_seed=seed,
                              early_stopping_rounds=50, verbose=0,
                              task_type='GPU', devices='0')
        m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=(X.iloc[vi], y_tr_sqrt.iloc[vi]))
        oof_tmp[vi] = decode_sqrt(m.predict(X.iloc[vi]))
        test_tmp   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    oof_cb_ms  += oof_tmp  / len(CB_SEEDS)
    test_cb_ms += test_tmp / len(CB_SEEDS)
    print(f'  Seed {seed}: OOF MAE = {mean_absolute_error(y, oof_tmp):.4f}')
print(f'M12 OOF MAE: {mean_absolute_error(y, oof_cb_ms):.4f}')

M12: CB Multi-seed (3 seeds)  [v8 신규]


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  Seed 42: OOF MAE = 8.8034


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  Seed 777: OOF MAE = 8.7979


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU


  Seed 2025: OOF MAE = 8.8014
M12 OOF MAE: 8.7982


## 19. 앙상블 — SLSQP (12개 모델)

In [20]:
oofs  = [oof_lgb_sqrt, oof_cb_sqrt, oof_xgb_mae, oof_lgb_log,
          oof_cb_log, oof_lgb_pow, oof_lgb_huber, oof_lgb_ms,
          oof_lgb_q75, oof_lgb_q85, oof_lgb_tw, oof_cb_ms]
tests = [test_lgb_sqrt, test_cb_sqrt, test_xgb_mae, test_lgb_log,
          test_cb_log, test_lgb_pow, test_lgb_huber, test_lgb_ms,
          test_lgb_q75, test_lgb_q85, test_lgb_tw, test_cb_ms]
names = ['LGB(sqrt)','CB(sqrt)','XGB(MAE)','LGB(log)',
         'CB(log)','LGB(pow)','LGB(Huber)','LGB(ms)',
         'LGB(q75)','LGB(q85)','LGB(Tweedie)','CB(ms)']
n_m = len(oofs)

print('=== 개별 모델 OOF MAE ===')
for name, oof in zip(names, oofs):
    mark = ' ←베스트' if mean_absolute_error(y, oof) == min(mean_absolute_error(y, o) for o in oofs) else ''
    print(f'  {name:>16}: {mean_absolute_error(y, oof):.4f}{mark}')

# SLSQP 최적화
def ens_loss(w): return mean_absolute_error(y, sum(w[i]*oofs[i] for i in range(n_m)))
res = minimize(ens_loss, [1/n_m]*n_m, method='SLSQP',
               bounds=[(0,1)]*n_m,
               constraints={'type':'eq','fun':lambda w:sum(w)-1})
weights_slsqp = res.x
print(f'\nSLSQP OOF MAE: {res.fun:.4f}')
print('SLSQP 가중치 (w>0.01):')
for n, w in sorted(zip(names, weights_slsqp), key=lambda x:-x[1]):
    if w > 0.01: print(f'  {n:>16}: w={w:.3f}')
print(f'\nv6 기준 개선: {8.7129 - res.fun:+.4f}')

=== 개별 모델 OOF MAE ===
         LGB(sqrt): 8.7809
          CB(sqrt): 8.8034
          XGB(MAE): 8.7756
          LGB(log): 8.7687
           CB(log): 8.7624
          LGB(pow): 8.7678
        LGB(Huber): 8.7408 ←베스트
           LGB(ms): 8.7627
          LGB(q75): 10.5623
          LGB(q85): 13.3266
      LGB(Tweedie): 8.9782
            CB(ms): 8.7982

SLSQP OOF MAE: 8.7212
SLSQP 가중치 (w>0.01):
        LGB(Huber): w=0.391
           CB(log): w=0.350
      LGB(Tweedie): w=0.132
          XGB(MAE): w=0.067
          LGB(log): w=0.059

v6 기준 개선: -0.0083


## 20. Tail 분석 + 후처리

In [21]:
oof_base = sum(weights_slsqp[i]*oofs[i] for i in range(n_m)).clip(0)
raw_test = sum(weights_slsqp[i]*tests[i] for i in range(n_m)).clip(0)

qs = np.arange(0.01, 1.00, 0.01)
q_true = np.quantile(y, qs)
q_oof  = np.quantile(oof_base, qs)

print('=== 분위수 비교 ===')
for p in [.50,.75,.90,.95,.99]:
    i = int(p*100)-1
    print(f'  {p:.0%}: 실제={q_true[i]:6.1f}, 예측={q_oof[i]:6.1f}, 차이={q_oof[i]-q_true[i]:+.1f}')

# Tail-aware 보정 (Quantile 모델 블렌딩)
oof_q_blend  = 0.5 * oof_lgb_q75  + 0.5 * oof_lgb_q85
test_q_blend = 0.5 * test_lgb_q75 + 0.5 * test_lgb_q85

tail_threshold = np.percentile(oof_base, 80)
tail_alpha_oof  = np.clip((oof_base  - tail_threshold) / (tail_threshold + 1), 0, 0.35)
tail_alpha_test = np.clip((raw_test  - tail_threshold) / (tail_threshold + 1), 0, 0.35)

oof_tail  = (1 - tail_alpha_oof)  * oof_base  + tail_alpha_oof  * oof_q_blend
test_tail = (1 - tail_alpha_test) * raw_test  + tail_alpha_test * test_q_blend

mae_base = mean_absolute_error(y, oof_base)
mae_tail = mean_absolute_error(y, oof_tail)
print(f'\nTail 보정: {mae_base:.4f} → {mae_tail:.4f} ({mae_tail-mae_base:+.4f})')

if mae_tail < mae_base:
    final_preds = test_tail.clip(0)
    oof_final   = oof_tail
    print('✅ Tail 보정 적용')
else:
    final_preds = raw_test.clip(0)
    oof_final   = oof_base
    print('❌ Tail 보정 미적용 (기본 SLSQP 선택)')

print('\n=== 최종 예측 분포 ===')
print(f'  mean={final_preds.mean():.2f}, std={final_preds.std():.2f}')
print(f'  95th={np.percentile(final_preds,95):.1f}, 99th={np.percentile(final_preds,99):.1f}, max={final_preds.max():.1f}')
print(f'  v6 99th=47.9 → v8 99th={np.percentile(final_preds,99):.1f}')

=== 분위수 비교 ===
  50%: 실제=   9.0, 예측=   8.9, 차이=-0.2
  75%: 실제=  25.8, 예측=  31.0, 차이=+5.2
  90%: 실제=  45.2, 예측=  36.1, 차이=-9.2
  95%: 실제=  60.8, 예측=  38.3, 차이=-22.5
  99%: 실제= 120.9, 예측=  46.2, 차이=-74.7

Tail 보정: 8.7212 → 8.7315 (+0.0103)
❌ Tail 보정 미적용 (기본 SLSQP 선택)

=== 최종 예측 분포 ===
  mean=19.44, std=14.30
  95th=39.8, 99th=48.6, max=68.4
  v6 99th=47.9 → v8 99th=48.6


## 21. Feature Importance

In [22]:
imp_df = pd.DataFrame({'feature':feature_cols,'importance':lgb_imp}).sort_values('importance',ascending=False).reset_index(drop=True)
print('=== Top 30 Features ===')
for _, row in imp_df.head(30).iterrows():
    print(f"  {row['feature']:50s} {row['importance']:8.1f}")

v8_new = ['multi_bottleneck_score','extreme_congestion_flag','compounding_bottleneck',
           'robot_health_index','pack_dock_bottleneck','global_load_stress',
           'dock_pressure_composite','slot_progress_x_order_load',
           'slot_progress_x_congestion','slot_progress_x_pack_util',
           'sc_double_bottleneck','sc_pack_x_progress','sc_congestion_x_prog']
print('\n=== v8 신규 피처 중요도 ===')
for feat in v8_new:
    row = imp_df[imp_df['feature']==feat]
    if len(row):
        rank = row.index[0]+1
        imp  = row['importance'].values[0]
        print(f"  [{rank:3d}위] {feat:45s} {imp:8.1f}")

=== Top 30 Features ===
  sc_pack_utilization_mean                             1227.2
  sc_robot_utilization_mean                            1062.6
  sc_congestion_score_mean                             1053.8
  sc_loading_dock_util_mean                            1003.8
  avg_trip_distance                                     993.8
  sc_order_inflow_15m_mean                              986.0
  sc_battery_mean_mean                                  878.2
  sc_double_bottleneck                                  805.4
  layout_compactness                                    786.2
  robot_per_pack_station                                687.2
  zone_dispersion                                       654.0
  sku_concentration                                     622.0
  intersection_count                                    548.8
  aisle_width_avg                                       502.0
  one_way_ratio                                         491.0
  floor_area_sqm                              

## 22. 제출

In [23]:
sub = pd.read_csv('./data/sample_submission.csv').drop(columns=[TARGET], errors='ignore')
sub = sub.merge(pd.DataFrame({'ID':test_fe['ID'], TARGET:final_preds}), on='ID', how='left')
sub.to_csv('./submission_v8.csv', index=False)
print('submission_v8.csv 저장 완료')
print(sub[TARGET].describe())
try:
    v6 = pd.read_csv('./submission_v6.csv')
    print(f'\n=== v6 vs v8 비교 ===')
    print(f'  99th: {np.percentile(v6[TARGET],99):.1f} → {np.percentile(final_preds,99):.1f}')
    print(f'  max:  {v6[TARGET].max():.1f} → {final_preds.max():.1f}')
    print(f'  mean: {v6[TARGET].mean():.2f} → {final_preds.mean():.2f}')
except: pass

submission_v8.csv 저장 완료
count    50000.000000
mean        19.440695
std         14.303327
min          0.444684
25%          5.498430
50%         14.976687
75%         34.019245
max         68.399661
Name: avg_delay_minutes_next_30m, dtype: float64

=== v6 vs v8 비교 ===
  99th: 47.9 → 48.6
  max:  71.1 → 68.4
  mean: 19.24 → 19.44
